<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH08/CH08_NB02_KVCache_vLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# Quantizing the KV Cache: vLLM
## Production-grade KV cache quantization with FP8

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/🤗%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
* **Author:** Pere Martra
* **Models:** meta-llama/Llama-3.2-3B
* **Colab Environment:** T4 GPU (Free Tier)
* **Keys:**
  * KV Cache
  * vLLM
  * FP8
* **References:**
  * [Manning Book: Rearchitecting LLMs](https://livebook.manning.com/book/rearchitecting-llms)
  * [vLLM Documentation](https://docs.vllm.ai/)
_____

This notebook compares two vLLM inference configurations to demonstrate the effect of FP8 KV cache quantization:

1. **FP16 KV cache (baseline)** — standard vLLM configuration; keys and values stored in FP16.
2. **FP8 KV cache** — a single configuration change causes vLLM to store keys and values in FP8, doubling the number of KV cache blocks available for the same VRAM budget.

The central metric exposed by vLLM is **# GPU blocks**: the number of paged attention blocks pre-allocated at startup. More blocks means the engine can handle longer sequences and larger concurrent batches before evicting or pausing generation.

## ⚠️ Important: start from a clean session

This notebook must be run in a **fresh Colab session** with no other notebooks active.
vLLM pre-allocates most of the available GPU memory at startup. Any residual allocation
from a previous session will cause initialization to fail or reduce the number of KV cache
blocks, distorting the measurements.

If in doubt: **Runtime → Factory reset runtime** before running.

## 1. Install dependencies

In [ ]:
!pip install -q vllm

> **Note:** vLLM installs its own pinned versions of `torch` and `transformers`.
> This notebook intentionally installs nothing else — adding other libraries before vLLM
> can cause dependency conflicts in Colab.

## 2. Imports and configuration

In [ ]:
import re
import gc
import time
import logging

from vllm import LLM, SamplingParams

MODEL_NAME = "meta-llama/Llama-3.2-3B"

# Long prompt to make KV cache growth visible across both tests
PROMPT = (
    "Explain in detail the history of artificial intelligence, "
    "covering all major milestones from the 1950s to the present day."
)
MAX_NEW_TOKENS = 500

## 3. Helper: capture GPU blocks and throughput

vLLM logs the number of pre-allocated KV cache blocks at startup:

```
INFO ... # GPU blocks: 1234, # CPU blocks: 0
```

The helper below intercepts that log line so we can record the value programmatically
and include it in the final comparison table.

In [ ]:
class _GpuBlockHandler(logging.Handler):
    """Logging handler that scans vLLM log lines for the GPU blocks count."""
    def __init__(self):
        super().__init__()
        self.gpu_blocks = None

    def emit(self, record):
        msg = record.getMessage()
        m = re.search(r'#\s*GPU blocks:\s*(\d+)', msg)
        if m:
            self.gpu_blocks = int(m.group(1))


def run_vllm_test(llm: LLM, sampling_params: SamplingParams, handler: _GpuBlockHandler):
    """Generate with *llm* and return (gpu_blocks, throughput_tokens_s, output_text)."""
    start = time.perf_counter()
    outputs = llm.generate([PROMPT], sampling_params)
    elapsed = time.perf_counter() - start

    output_text = outputs[0].outputs[0].text
    n_tokens = len(outputs[0].outputs[0].token_ids)
    throughput = round(n_tokens / elapsed, 2)

    return handler.gpu_blocks, throughput, output_text


# Attach the handler to the vLLM root logger once; reuse across both tests
_handler = _GpuBlockHandler()
logging.getLogger("vllm").addHandler(_handler)

sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)

print("Helper ready.")

## 4. Test 1 — Baseline vLLM: FP16 KV cache

Standard vLLM launch with `dtype="float16"`. Keys and values are stored in FP16 inside
each paged attention block.

**What to watch in the startup logs:**  
The line `# GPU blocks: N` tells you how many KV cache blocks vLLM pre-allocated.
This number is our reference — Test 2 should be strictly larger.

In [ ]:
# Reset the handler before each test so it captures this run's block count
_handler.gpu_blocks = None

llm_baseline = LLM(
    model=MODEL_NAME,
    dtype="float16",
    gpu_memory_utilization=0.85,   # Required on T4 to prevent OOM
)

gpu_blocks_fp16, throughput_fp16, text_fp16 = run_vllm_test(
    llm_baseline, sampling_params, _handler
)

print(f"\n{'='*60}")
print(f"  GPU blocks (FP16) : {gpu_blocks_fp16}")
print(f"  Throughput        : {throughput_fp16} tokens/s")
print(f"{'='*60}\n")
print(text_fp16)

del llm_baseline
gc.collect()

**Expected:** `# GPU blocks` in the hundreds. This is the baseline capacity.
Each block stores 16 tokens of KV data in FP16 (2 bytes per value × 2 for K and V).

## 5. Test 2 — vLLM with FP8 KV cache

A single extra argument — `kv_cache_dtype="fp8"` — tells vLLM to store every key and
value tensor in FP8 (1 byte per value) instead of FP16 (2 bytes per value).

Because each block now occupies half the VRAM, vLLM can pre-allocate roughly **twice as
many blocks** from the same memory budget. The model weights and activations are still
computed in FP16; only the cached attention tensors change precision.

**What to watch:** `# GPU blocks` should be noticeably larger than in Test 1.

In [ ]:
# Reset the handler before each test
_handler.gpu_blocks = None

llm_fp8 = LLM(
    model=MODEL_NAME,
    kv_cache_dtype="fp8",          # KV cache quantized natively to FP8
    dtype="float16",
    gpu_memory_utilization=0.85,
)

gpu_blocks_fp8, throughput_fp8, text_fp8 = run_vllm_test(
    llm_fp8, sampling_params, _handler
)

print(f"\n{'='*60}")
print(f"  GPU blocks (FP8)  : {gpu_blocks_fp8}")
print(f"  Throughput        : {throughput_fp8} tokens/s")
print(f"{'='*60}\n")
print(text_fp8)

del llm_fp8
gc.collect()

**Expected:** `# GPU blocks` approximately double that of Test 1.
Throughput should be similar to — or slightly above — the FP16 baseline, because
FP8 is handled natively in vLLM's CUDA kernels with minimal overhead.

## 6. Results summary

The table below consolidates both tests. The key lever here is **# GPU blocks**:
with FP8, vLLM can serve longer contexts and larger batches from the same T4 memory budget.

| Lever | Affects GPU blocks? | Affects throughput? |
|---|---|---|
| FP16 KV cache (baseline) | — reference — | — reference — |
| FP8 KV cache (`kv_cache_dtype="fp8"`) | ✅ Yes — ~2× more blocks | ➡ Similar or slightly higher |

In [ ]:
import pandas as pd

# Derived metric: capacity gain as a percentage relative to FP16 baseline
capacity_gain_fp16 = 0.0
capacity_gain_fp8  = round((gpu_blocks_fp8 / gpu_blocks_fp16 - 1) * 100, 1)

summary = pd.DataFrame([
    {
        "Configuration":          "FP16 KV Cache (baseline)",
        "KV Cache dtype":         "float16",
        "GPU Blocks":             gpu_blocks_fp16,
        "Capacity Gain (%)":      capacity_gain_fp16,
        "Throughput (Tokens/s)":  throughput_fp16,
    },
    {
        "Configuration":          "FP8 KV Cache",
        "KV Cache dtype":         "fp8",
        "GPU Blocks":             gpu_blocks_fp8,
        "Capacity Gain (%)":      capacity_gain_fp8,
        "Throughput (Tokens/s)":  throughput_fp8,
    },
])

print(summary.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- Data ---
configurations = ['FP16 KV Cache\n(baseline)', 'FP8 KV Cache']

gpu_blocks  = [gpu_blocks_fp16,  gpu_blocks_fp8]
cap_gain    = [capacity_gain_fp16, capacity_gain_fp8]
throughput  = [throughput_fp16,  throughput_fp8]

colors  = ['lightblue', 'orange']
hatches = ['//', '\\\\']
x       = np.arange(len(configurations))
width   = 0.5

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.patch.set_facecolor('white')

panels = [
    (axes[0], gpu_blocks,  'GPU Blocks',             'GPU Blocks (KV Cache Capacity)', '{:.0f}'),
    (axes[1], cap_gain,    'Capacity Gain (%)',       'Capacity Gain vs FP16 Baseline', '{:.1f}%'),
    (axes[2], throughput,  'Throughput (Tokens/s)',   'Throughput (Tokens/s)',           '{:.1f}'),
]

for ax, values, ylabel, title, fmt in panels:
    ax.set_facecolor('white')
    max_val = max(values) if max(values) > 0 else 1
    for i, (val, color, hatch) in enumerate(zip(values, colors, hatches)):
        ax.bar(
            x[i], val, width,
            color=color, alpha=0.9,
            hatch=hatch, edgecolor='black'
        )
        ax.text(
            x[i], val + max_val * 0.02,
            fmt.format(val),
            ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='dimgray'
        )

    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(configurations, fontsize=9)
    ax.set_ylim(0, max_val * 1.20)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6, color='gray')
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

# --- Legend ---
legend_elements = [
    mpatches.Patch(facecolor='lightblue', hatch='//',   edgecolor='black', label='FP16 KV Cache (baseline)'),
    mpatches.Patch(facecolor='orange',    hatch='\\\\', edgecolor='black', label='FP8 KV Cache'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=10, framealpha=0.8, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('KV Cache Quantization with vLLM — FP16 vs FP8',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('kvcache_vllm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

The two tests isolate the effect of a single configuration knob — `kv_cache_dtype` — inside
vLLM's paged attention engine.

**GPU blocks** is the key capacity metric in vLLM. Each block holds a fixed number of
tokens (16 by default) worth of key and value tensors. With FP16, each entry occupies
2 bytes; with FP8, it occupies 1 byte. All else being equal, the FP8 run should
pre-allocate roughly **twice as many blocks** from the same VRAM budget — and that
extra capacity translates directly into the ability to serve longer sequences and larger
concurrent batches without swapping or pausing.

**Throughput** remains comparable between the two runs because vLLM implements FP8 KV
cache through native CUDA kernels. Unlike Python-level quantization backends (such as
Quanto in the companion notebook), the quantize/dequantize path in vLLM adds negligible
latency per token, so the generation speed is not penalized by the cache precision change.

**Contrast with the Hugging Face notebook (NB01):** the Quanto 4-bit KV cache backend
also reduced dynamic VRAM usage, but caused a ~6× throughput drop because the
quantization ran in Python on every attention step. vLLM's FP8 implementation avoids
that penalty entirely — this is the difference between a research-oriented backend and a
production-grade inference engine.

The two approaches are complementary: weight quantization (BitsAndBytes, GPTQ, AWQ)
shrinks the **static** model footprint; KV cache quantization (FP8 in vLLM) shrinks the
**dynamic** footprint that grows with sequence length. In production, both levers can be
applied simultaneously.